In [ ]:
#| default_exp Environments/SocialDilemmaAsymJ

"""Class for an asymmetric two-agent stateless social dilemma environment with agent-specific payoffs"""

# AUTOGENERATED! DO NOT EDIT! File to edit: ../../nbs/Environments/101_EnvSocialDilemmaJ.ipynb.



In [ ]:
#| export
__all__ = ['SocialDilemmaAsymJ']




In [ ]:
#| export
from fastcore.foundation import patch
from pyCRLD.Environments.BaseJ import ebaseJ

from fastcore.utils import *
from fastcore.test import *

import jax.numpy as jnp




In [ ]:
#| export
class SocialDilemmaAsymJ(ebaseJ):
    """
    JAX-friendly version of a 2-agent 2-action Social Dilemma Matrix Game.
    Supports asymmetric payoffs for each agent.
    """ 
    def __init__(self,
                 payoffs: dict = None,  # dict with 'agent_0' and 'agent_1' payoff dicts
                 R: float = None,  # reward of mutual cooperation (symmetric case)
                 T: float = None,  # temptation of unilateral defection (symmetric case)
                 S: float = None,  # sucker's payoff of unilateral cooperation (symmetric case)
                 P: float = None   # punishment of mutual defection (symmetric case)
                ):
        """
        Initialize either with:
        1. Symmetric game: R, T, S, P (both agents have same payoffs)
        2. Asymmetric game: payoffs dict like:
           {
               'agent_0': {'R': r0, 'T': t0, 'S': s0, 'P': p0},
               'agent_1': {'R': r1, 'T': t1, 'S': s1, 'P': p1}
           }
        """
        self.N = 2
        self.M = 2
        self.Z = 1

        # Handle symmetric vs asymmetric initialization
        if payoffs is not None:
            # Asymmetric case
            self.asymmetric = True
            self.payoffs = payoffs
            # Store individual agent payoffs
            self.agent_0_payoffs = payoffs['agent_0']
            self.agent_1_payoffs = payoffs['agent_1']
        else:
            # Symmetric case - use provided R, T, S, P
            self.asymmetric = False
            self.Re = R
            self.Te = T
            self.Su = S    
            self.Pu = P
            # Create symmetric payoff structure
            self.payoffs = {
                'agent_0': {'R': R, 'T': T, 'S': S, 'P': P},
                'agent_1': {'R': R, 'T': T, 'S': S, 'P': P}
            }
            self.agent_0_payoffs = self.payoffs['agent_0']
            self.agent_1_payoffs = self.payoffs['agent_1']

        self.state = 0  # initial state
        super().__init__()





In [ ]:
#| export
@patch
def TransitionTensor(self: SocialDilemmaAsymJ):
    """Get the Transition Tensor."""
    # Return a JAX array of ones with shape (Z, M, M, Z)
    Tsas = jnp.ones((self.Z, self.M, self.M, self.Z))
    return Tsas




In [ ]:
#| export
@patch
def RewardTensor(self: SocialDilemmaAsymJ):
    """Get the Reward Tensor R[i,s,a1,...,aN,s'] with agent-specific payoffs."""
    R = jnp.zeros((2, self.Z, 2, 2, self.Z))
    
    # Agent 0's payoff matrix (rows = agent 0's actions, cols = agent 1's actions)
    # When agent 0 cooperates and agent 1 cooperates: R
    # When agent 0 cooperates and agent 1 defects: S (sucker)
    # When agent 0 defects and agent 1 cooperates: T (temptation)
    # When agent 0 defects and agent 1 defects: P (punishment)
    p0 = self.agent_0_payoffs
    R = R.at[0, 0, :, :, 0].set(jnp.array([[p0['R'], p0['S']],
                                             [p0['T'], p0['P']]]))
    
    # Agent 1's payoff matrix (rows = agent 0's actions, cols = agent 1's actions)
    # When agent 0 cooperates and agent 1 cooperates: R
    # When agent 0 cooperates and agent 1 defects: T (temptation for agent 1)
    # When agent 0 defects and agent 1 cooperates: S (sucker for agent 1)
    # When agent 0 defects and agent 1 defects: P (punishment)
    p1 = self.agent_1_payoffs
    R = R.at[1, 0, :, :, 0].set(jnp.array([[p1['R'], p1['T']],
                                             [p1['S'], p1['P']]]))
    return R




In [ ]:
#| export
@patch
def actions(self:SocialDilemmaAsymJ):
    """The action sets"""
    return [['c', 'd'] for _ in range(self.N)]




In [ ]:
#| export
@patch
def states(self:SocialDilemmaAsymJ):
    """The states set"""
    return ['.'] 




In [ ]:
#| export
@patch
def id(self:SocialDilemmaAsymJ):
    """
    Returns id string of environment
    """
    if self.asymmetric:
        # For asymmetric games, include both agents' payoffs
        p0 = self.agent_0_payoffs
        p1 = self.agent_1_payoffs
        id = f"{self.__class__.__name__}_asym_" + \
            f"A0_T{p0['T']}_R{p0['R']}_P{p0['P']}_S{p0['S']}_" + \
            f"A1_T{p1['T']}_R{p1['R']}_P{p1['P']}_S{p1['S']}"
    else:
        # Symmetric case
        id = f"{self.__class__.__name__}_sym_" + \
            f"{self.Te}_{self.Re}_{self.Pu}_{self.Su}"
    return id
